# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Categorized `job_title` into a new `job_category` feature to reduce cardinality.
    * Engineered 10 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [1]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [2]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [3]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [4]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [5]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [6]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [7]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

print(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh', 'kabacehsingkil': 'Aceh', 'kabacehselatan': 'Aceh', 'kabacehtenggara': 'Aceh', 'kabacehtimur': 'Aceh', 'kabacehtengah': 'Aceh', 'kabacehbarat': 'Aceh', 'kabacehbesar': 'Aceh', 'kabpidie': 'Aceh', 'kabbireuen': 'Aceh', 'kabacehutara': 'Aceh', 'kabacehbaratdaya': 'Aceh', 'kabgayolues': 'Aceh', 'kabacehtamiang': 'Aceh', 'kabnaganraya': 'Aceh', 'kabacehjaya': 'Aceh', 'kabbenermeriah': 'Aceh', 'kabpidiejaya': 'Aceh', 'kotabandaaceh': 'Aceh', 'kotasabang': 'Aceh', 'kotalangsa': 'Aceh', 'kotalhokseumawe': 'Aceh', 'kotasubulussalam': 'Aceh', 'kabnias': 'Sumatera Utara', 'kabmandailingnatal': 'Sumatera Utara', 'kabtapanuliselatan': 'Sumatera Utara', 'kabtapanulitengah': 'Sumatera Utara', 'kabtapanuliutara': 'Sumatera Utara', 'kabtobasamosir': 'Sumatera Utara', 'kablabuhanbatu': 'Sumatera Utara', 'kabasahan': 'Sumatera Utara', 'kabsimalungun': 'Sumatera Utara', 'kabdairi': 'Sumatera Utara', 'kabkaro': 'Sumatera Utara', 'kabdeliserdang': 'Sumatera Utara', 'kablangkat': 'Su

In [8]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [9]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [10]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [11]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

In [12]:
# Fix some regency and city names
internship_positions["regency_city"] = (
    internship_positions.regency_city
    .str.replace(r"^Kab\s", r"Kab. ", regex=True)
    .str.replace("Pahuwato", "Pohuwato")
    .str.replace(r"^Kepulauan\s", r"Kab. Kep. ", regex=True)
    .str.replace(r"\sKepulauan\s", r" Kep. ", regex=True)
)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [13]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [14]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
21499,a240ef28-5d71-4d73-9f53-d2ef944a9805,2026-07-16T12:46:34+07:00,PEMBINAAN KEPRIBADIAN,LEMBAGA PEMASYARAKATAN KELAS III PAGAR ALAM,Kota Pagar Alam,Bachelor,Seni Tari,1.\tMenyusun dan melaksanakan program pembinaa...,6,1,1,12,Sumatera Selatan,7.69
27428,a2415bcb-ce51-496c-9193-ad6af6c23121,2026-07-16T12:12:46+07:00,PEMBINAAN KEPRIBADIAN,LEMBAGA PEMASYARAKATAN KELAS IIA KEDIRI,Kota Kediri,Bachelor,"Pendidikan Agama Islam (PAI), Pendidikan Seni ...","""1.\tMenyusun dan melaksanakan program pembina...",6,2,2,51,Jawa Timur,3.85
13018,a23f5da3-7dd3-4db4-b028-72d4ed655c9f,2026-07-16T12:12:30+07:00,PENGELOLA KEHUMASAN,LEMBAGA PEMASYARAKATAN KELAS IIA KALIANDA,Kab. Lampung Selatan,Bachelor,Komunikasi,1. Menyusun materi layanan informasi untuk med...,6,1,1,7,Lampung,12.50
23999,a241080a-9d35-4d84-ac3c-37d28417c758,2026-07-16T12:58:48+07:00,PENGELOLA KEGIATAN KERJA,BALAI PEMASYARAKATAN KELAS I PANGKAL PINANG,Kota Pangkal Pinang,Bachelor,"Agribinis, Manajemen, Perikanan, Teknologi Has...",1. Menyusun rencana program kegiatan kerja dan...,5,1,1,15,Kepulauan Bangka Belitung,6.25
25398,a23f4940-f0ca-424d-998d-eaa9a3b69bd9,2026-07-16T10:14:52+07:00,Quality and Risk Staff Intern,PT Bina Bahtera Sejati (Rumah Sakit Siloam Buton),Kota Bau Bau,Bachelor,"Keperawatan, Kesehatan Masyarakat, Ilmu Kepera...",Membantu QR staff dalam kegiatan pengawasan mu...,5,1,1,17,Sulawesi Tenggara,5.56
20123,a240d10d-835b-4b27-ac29-fbccb6c669c7,2026-07-16T10:59:03+07:00,ADMINISTRASI SALES,Perusahaan Perseroan (Persero) Bank Negara Ind...,Kab. Jember,"Bachelor, Diploma","Ilmu Komputer, Pengelolaan Arsip dan Rekaman I...","1) Mempelajari pengelolaan data, dokume...",5,3,3,32,Jawa Timur,9.09
20203,a23fb969-47cc-4ba8-a7f0-b1d70172b39b,2026-07-16T15:51:15+07:00,Supply Chain Management,Kimia Farma Apotek,Kota Adm. Jakarta Pusat,"Bachelor, Diploma","Teknik Informatika, Manajemen, Teknik Industri...",1. Membantu menarik data pembelian untuk mengh...,5,1,1,11,DKI Jakarta,8.33
18520,a2411e0f-2da4-4b14-9ea5-b172398e0372,2026-07-16T12:35:21+07:00,Asisten Pengelola Keuangan,BPS Kabupaten Kendal,Kab. Kendal,Bachelor,"Manajemen Keuangan, Perpajakan, Manajemen, Aku...",Membantu proses pengelolaan administrasi keuan...,5,2,2,19,Jawa Tengah,10.00
18524,a240c20a-11ac-4eb2-87ba-1227003b324c,2026-07-16T12:34:15+07:00,DUTA LAYANAN,KANIM KELAS II TPI SUMBAWA BESAR,Kab. Sumbawa,Bachelor,"Komunikasi, Hubungan Masyarakat, Pariwisata","""1. Memberikan pelayanan langsung kepada masya...",5,2,2,19,Nusa Tenggara Barat,10.00
18281,a2376024-36ea-467a-87d1-474ad54a7c00,2026-07-16T11:01:56+07:00,Staf Pengembangan Budaya Perusahaan,PT Perusahaan Perseroan (Persero) PT. Pos Indo...,Kota Bandung,Bachelor,Desain Komunikasi Visual,"Merencanakan, menyusun, dan memproduksi berbag...",5,2,2,18,Jawa Barat,10.53


## 3.2 Feature Transformation
### 3.2.1 Text Classification
* Categorizing `job_title` into a new `job_category` feature to reduce cardinality.

In [15]:
# Create a custom function to categorize the jobs
def categorize_job(title):
    if pd.isna(title):
        return "Uncategorized"
    
    t = str(title).lower()
    
    # 1. Healthcare & Medical (Added severe typos, hospital codes, and specialized terms)
    if any(w in t for w in ["perawat", "ners", "nurse", "psikiat", "spikiat", "psikat", "pskiat", "psikolog", "piskolog", "pskolog", "medis", "medic", "medik", "gizi", "nutri", "diet", "dokter", "doker", "doktor", "bidan", "apotek", "aptoker", "farmasi", "pharmac", "fisio", "physio", "radio", "sanitari", "sanitasi", "kesehatan", "promkes", "okupasi", "elektromedis", "atem", "terapi", "therap", "klinik", "atlm", "epidemiolog", "anestesi", "anastesi", "rekam medi", "perekam", "mr ", "casemix", "coder", "koder", "cssd", "ipsrs", "audiolog", "orthotic", "mcu", "ranap", "igd", "poliklinik", "vk ", "bersalin", "hemodialisa", "kardiovaskuler", "cardiovascular", "refraksi", "kebidanan", "keperawatan", "patologi", "mikrobiologi", "imunologi", "darah", "ambul", "hospital", "rehabilitasi", "admission"]):
        return "Healthcare & Medical"
    
    # 2. IT & Data
    elif any(w in t for w in ["komputer", "it ", " it", "programmer", "developer", "software", "data", "sistem", "system", "ui/", "/ux", "network", "cyber", "aplikasi", "application", "backend", "frontend", "website", "web", "informatika", "pusdatin", "jaringan", "ai ", "machine learning", "bda", "digital", "erp", "sap ", "helpdesk", "support it", "noc ", "rpa ", "command center", "dashboard", "cloud", "iot", "analytic"]):
        return "IT & Data"
        
    # 3. Engineering & Maintenance
    elif any(w in t for w in ["teknis", "technician", "maintenance", "engineer", "mekanik", "mechanic", "drafter", "drawing", "listrik", "sipil", "civil", "bangunan", "hvac", "otomotif", "mesin", "machine", "welder", "welding", "proyek", "project", "elektro", "electric", "arsitek", "architect", "maint", "equipment", "facility", "sarana", "prasarana", "geologi", "tambang", "mining", "instrument", "surveyor", "craft", "plumbing", "geofisika", "seismik", "geodesi", "geomatika", "toolman", "inspector", "inspektur", "konstruksi", "construction", "pipa", "baja", "otomasi", "automation"]):
        return "Engineering & Maintenance"
        
    # 4. Manufacturing, QA & Production
    elif any(w in t for w in ["produksi", "production", "operator", "qc", "qa", "quality", "pabrik", "manufacturing", "assembly", "packaging", "mutu", "plant", "molding", "mould", "mold", "pattern maker", "slitting", "blown film", "sewing", "garment", "textile", "printing", "finishing", "curing", "mixing", "extruder", "ppic", "rnd", "r&d", "research", "reserch", "set up", "cleanning", "rewinding", "laminasi", "improvement", "pdca", "koe ", "lean", "mill", "helper", "pe ", "ie "]):
        return "Manufacturing, QA & Production"
        
    # 5. Finance & Banking
    elif any(w in t for w in ["keuangan", "akuntansi", "accounting", "akuntan", "pajak", "tax", "auditor", "audit", "bendahara", "anggaran", "finance", "treasury", "billing", "kasir", "cashier", "credit", "kredit", "loan", "pembiayaan", "funding", "transaction", "collection", "receivable", "payable", "wealth", "insurance", "asuransi", "actuary", "aktuaria", "bank", "bni", "teller", "pawning", "micro", "invest", "budget", "cost "]):
        return "Finance & Banking"
        
    # 6. Sales, Marketing & Hospitality
    elif any(w in t for w in ["barista", "cook", "koki", "pastry", "bakery", "culinary", "chef", "layanan", "frontliner", "frontlner", "sales", "marketing", "f&b", "fb ", "store", "customer", "receptionist", "pemasaran", "pramusaji", "hotel", "event", "reservation", "guest", "hospitality", "dancer", "entertainment", "commercial", "merchandis", "retail", "promot", "promosi", "brand", "business development", "bd ", "partnership", "account executive", "masak", "food", "beverage", "catering", "kitchen", "tour ", "travel", "room", "bro", "activation"]):
        return "Sales, Marketing & Hospitality"
        
    # 7. Media, PR & Creative
    elif any(w in t for w in ["humas", "kehumasan", "design", "desain", "kreatif", "creative", "video", "animator", "animation", "content", "konten", "sosial media", "social media", "sosmed", "kol ", "publisitas", "jurnalis", "journalist", "wartawan", "editor", "multimedia", "reporter", "fotografer", "photographer", "broadcasting", "komunikasi", "communication", "visual", "vm artist", "motion", "copywriter", "writer", "art ", "talent", "audio", "camera", "campaign", "publikasi", "publik", "illustrator", "media", "broadcast", "creator"]):
        return "Media, PR & Creative"
        
    # 8. Legal, Risk & Compliance
    elif any(w in t for w in ["hukum", "legal", "law", "compliance", "kepatuhan", "risk", "risiko", "hse", "hsse", "qhse", "she", "ehs", "safety", "k3", "security", "keamanan", "fraud", "investigasi", "pengaduan", "maladministrasi", "litigasi", "regulas", "regulatory", "sertifikasi", "perizinan", "izin", "kekayaan intelektual"]):
        return "Legal, Risk & Compliance"
        
    # 9. Logistics & Supply Chain
    elif any(w in t for w in ["gudang", "warehouse", "logistik", "logistic", "exim", "supply chain", "scm", "inventory", "pengadaan", "purchasing", "procurement", "buyer", "ekspor", "impor", "export", "import", "cargo", "shipping", "freight", "delivery", "transport", "fleet", "ekspeditor", "harbour", "port ", "bandara", "airport", "pelabuhan", "aviation", "aero", "aircraft", "checker", "terminal"]):
        return "Logistics & Supply Chain"
        
    # 10. Education, Training & Government
    elif any(w in t for w in ["kebijakan", "pemerintahan", "penelaah", "pengawas", "asn", "biro", "kementerian", "pemda", "pns", "diplomat", "instruktur", "pelatihan", "pembelajaran", "tentor", "pengajar", "edukator", "diklat", "akademik", "guru", "dosen", "widyaiswara", "pusat", "badan", "tutor", "statistik", "statistisi", "peneliti", "pustaka", "kearsipan", "arsip", "kurator", "laporan", "pelaporan", "penyusun", "evaluasi", "pengolah", "dokumen", "evaluator", "pemeriksaan"]):
        return "Education, Training & Government"
        
    # 11. Agriculture & Environment
    elif any(w in t for w in ["pertanian", "perikanan", "peternakan", "perkebunan", "agribisnis", "kehutanan", "lingkungan", "agronomi", "pangan", "tambak", "tanaman", "kebun", "hewan", "hutan", "forestry", "environment", "sustainability", "esg", "limbah", "waste", "marine", "hydro", "iklim", "climate", "budidaya", "ternak", "nelayan", "satwa", "flora", "fauna", "ekologi", "air "]):
        return "Agriculture & Environment"
        
    # 12. Language & Translation
    elif any(w in t for w in ["isyarat", "penerjemah", "translator", "interpreter", "language", "mandarin", "japanese", "english", "bahasa"]):
        return "Language & Translation"
        
    # 13. Correctional & Social Services
    elif any(w in t for w in ["pembinaan", "kepribadian", "pembimbing kemasyarakatan", "warga binaan", "kegiatan kerja", "rohani", "sosial", "pemasyarakatan", "community", "csr", "tjsl", "bina", "klien", "konselor"]):
        return "Correctional & Social Services"

    # 14. HR, Admin & Management (General catch-alls placed at the very end)
    elif any(w in t for w in ["sdm", "human resource", "hr", "ga", "general affair", "administrasi", "admin", "bmn", "sekretaris", "secretary", "tata usaha", "tu ", "personil", "personalia", "rekrutmen", "recruitment", "talent acquisition", "od ", "organization development", "umum", "fasilitas", "manajemen", "management", "manager", "pmo", "strategi", "koordinator", "coordinator", "supervisor", "spv", "director", "operasional", "operation", "asset", "aset", "clerical", "sarana", "pejabat", "pengelola", "asisten", "officer", "staff", "staf", "pelaksana", "magang", "intern", "consultant", "konsultan", "planner"]):
        return "HR, Admin & Management"
        
    else:
        return "Other"

In [16]:
# Apply the function to create the new column
internship_positions["job_category"] = internship_positions["job_title"].apply(categorize_job)

# Verify the distribution of the new categories
display(internship_positions["job_category"].value_counts())

job_category
HR, Admin & Management              5176
Healthcare & Medical                4063
IT & Data                           2802
Media, PR & Creative                2570
Sales, Marketing & Hospitality      2150
Correctional & Social Services      2005
Finance & Banking                   1707
Engineering & Maintenance           1701
Education, Training & Government    1294
Manufacturing, QA & Production      1083
Legal, Risk & Compliance            1036
Other                                994
Logistics & Supply Chain             837
Agriculture & Environment            597
Language & Translation               307
Name: count, dtype: int64

* Engineering 9 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.

In [17]:
# Create new Boolean columns
# Define the categories and their specific Indonesian Regex keywords
maj_categories = {
    "it_and_computer": r"informatika|komputer|sistem informasi|perangkat lunak|multimedia|jaringan|siber|data|teknologi informasi|piranti lunak|website",
    "engineering": r"teknik(?!\s*(?:informatika|komputer|multimedia))|rekayasa(?!\s*(?:perangkat lunak|internet|komputer))|arsitektur|mesin|elektro|sipil|industri|mekatronika|otomotif|manufaktur|konstruksi|geodesi|geologi|tambang|perkapalan|dirgantara|nautika|listrik|kelistrikan|logam|tekstil|metrologi|instrumentasi|perencanaan|planologi|tata ruang",
    "business": r"manajemen|akuntansi|bisnis|ekonomi|keuangan|administrasi|adminsitrasi|logistik|pemasaran|marketing|pajak|perbankan|retail|niaga|aktiva|kewirausahaan|asuransi",
    "health": r"kedokteran|keperawatan|kebidanan|farmasi|kesehatan|gizi|medik|medis|terapi|radiologi|klinik|apoteker|sanitasi|higiene|hiperkes|optisi|optometri|ortotik|prostetik|darah|audiologi|akupunktur|herbal|rumah sakit|nutrisi",
    "science_and_math": r"matematika|statistik|statistika|biologi|kimia|fisika|sains|aktuaria|geografi|astronomi|lingkungan|bumi|kartografi|penginderaan|oseanografi",
    "agriculture_and_fisheries": r"agribisnis|agribinis|pertanian|peternakan|perikanan|kehutanan|agroteknologi|agroekoteknologi|agro|perkebunan|agronomi|hortikultura|hewan|laut|budidaya|tanaman|pangan|pertanahan",
    "arts_and_media": r"desain|seni|komunikasi|film|televisi|jurnalistik|penyiaran|broadcasting|hubungan masyarakat|humas|fotografi|kriya|tari|musik|karawitan|animasi|media|audio|video|penerbitan",
    "social_and_law": r"hukum|sosiologi|psikologi|sastra|bahasa|kriminologi|kesejahteraan|pemerintahan|politik|hubungan internasional|sejarah|filsafat|antropologi|perpustakaan|kearsipan|arsip|agama|teologi|syariah|islam|kristen|buddha|hindu",
    "education": r"pendidikan|pgsd|pgpaud|tadris|bimbingan|konseling|tarbiyah|guru|kependidikan|penyuluhan",
    "tourism_and_hospitality": r"pariwisata|perhotelan|tata boga|tata rias|tata busana|fashion|kuliner|wisata|mice|travel|hospitaliti|hidang|patiseri"
}

# Ensure the column is treated as a string and handle missing values
internship_positions["allowed_major"] = internship_positions["allowed_major"].fillna("")

# Iterate through the dictionary to create the new Boolean columns
for cat, pattern in maj_categories.items():
    col_name = f"allows_{cat}_majors"
    
    # Check if any keyword in the pattern exists in the "allowed_major" string
    mask = internship_positions["allowed_major"].str.contains(pattern, case=False, regex=True)    
    internship_positions[col_name] = np.where(mask, "Yes", "No")

# Preview the results
display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_it_and_computer_majors,allows_engineering_majors,allows_business_majors,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors
3090,a23f3a20-019b-41ba-a7ba-ae03c2e3442e,2026-07-16T10:57:25+07:00,Marketing/ Sales Coordinator,Cahaya Buana Intitama,Kab. Bogor,"Diploma, Bachelor",Manajemen Pemasaran/Marketing,Penjualan / Promosi\n\n • Mengembangkan are...,6,2,...,No,No,Yes,No,No,No,No,No,No,No
6951,a243f274-f31b-4332-a1fa-49d2cfc30a57,2026-07-16T10:38:43+07:00,Market Research Intern (Quantitative),PT. Neurosensum Technology International,Kota Adm. Jakarta Selatan,Bachelor,"Manajemen, Psikologi, Bisnis, Ekonomi, Statistika",Mendukung pelaksanaan proses penelitian kuanti...,5,1,...,No,No,Yes,No,Yes,No,No,Yes,No,No
27050,a2411389-950f-4f02-bb1f-b5f385f640ff,2026-07-16T12:02:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB GRESIK,Kab. Gresik,Bachelor,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,5,1,...,No,No,No,No,No,No,No,Yes,No,No
15593,a2441362-2f95-4afd-8deb-f81993b31ed2,2026-07-16T12:04:56+07:00,Pengolah Data,"Kementerian Usaha Mikro, Kecil, dan Menengah",Kota Adm. Jakarta Selatan,"Bachelor, Diploma","Sains Data, Administrasi BIsnis, Administrasi,...","1. Pengumpulan, verifikasi, pengolahan, dan pe...",5,1,...,Yes,No,Yes,No,Yes,No,No,No,No,No
11395,a243b27c-516e-4d36-a80e-3a9f5cb56192,2026-07-16T12:09:57+07:00,DPEP - Pengawas DD DPE3.2,OJKI,Kota Adm. Jakarta Pusat,Bachelor,Akuntansi,"""Divisi Pengawas Emiten dan Perusahaan Publik ...",5,2,...,No,No,Yes,No,No,No,No,No,No,No
1155,a2437d4a-1728-42a5-9069-73833d3d05ab,2026-07-16T09:47:09+07:00,Guest Relation Officer - Mandarin Speaking,PT. Setia Meranti,Kota Adm. Jakarta Pusat,"Diploma, Bachelor, Profession","Bahasa Inggris Untuk Industri Pariwisata, Baha...",Mampu memberikan pelayanan terbaik kepada tamu...,5,1,...,No,Yes,Yes,No,No,No,Yes,Yes,No,Yes
9213,a2254934-7153-4de6-ad59-a51fcd45a3fa,2026-07-16T09:55:18+07:00,AI Engineer,"PT. Bali Towerindo Sentra, Tbk.",Kota Adm. Jakarta Pusat,Bachelor,"Teknik Informatika, Sains Data, Ilmu Komputer,...","Peserta magang membantu proses pengembangan, p...",5,5,...,Yes,Yes,No,No,Yes,No,No,No,No,No
21186,a24189d8-3431-4779-a0ba-63b273551583,2026-07-16T09:59:43+07:00,Administrasi,Agung Automall,Kota Jambi,"Diploma, Bachelor","Ekonomi Pembangunan, Akuntansi Keuangan Perusa...","Memproses pengelolaan administrasi keuangan, p...",6,5,...,No,No,Yes,No,No,No,No,No,No,No
15504,a23f9979-b5e9-4933-89c4-9260757d06eb,2026-07-16T12:13:12+07:00,PENGELOLA KEGIATAN KERJA,LEMBAGA PEMASYARAKATAN KELAS IIA KENDARI,Kota Kendari,Bachelor,Budidaya Perkebunan,$24,6,1,...,No,No,No,No,No,Yes,No,No,No,No
9406,a23f6d5d-fb52-4390-aa10-ed381528b541,2026-07-16T13:01:45+07:00,PENGELOLA SDM,RUMAH TAHANAN NEGARA KELAS IIB SANGGAU,Kab. Sanggau,Bachelor,Manajemen,1. Mengumpulkan data dan informasi yang releva...,6,1,...,No,No,Yes,No,No,No,No,No,No,No


In [18]:
# Review the rows that slipped through the Regex patterns
category_cols = [f"allows_{cat}_majors" for cat in maj_categories.keys()]
uncategorized_mask = (internship_positions[category_cols] == "No").all(axis=1)

uncategorized_positions = internship_positions[uncategorized_mask]

print(uncategorized_positions['allowed_major'].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str


### 3.2.2 Binning (Discretization)
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [19]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_business_majors,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,No,Yes,No,No,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2


In [20]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5


In [21]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [22]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
8272,a243ee6d-9348-41ba-8755-1ecc7eb05c45,2026-07-16T10:13:21+07:00,Digital Assistance & Prepaid Development Inter...,Perusahaan Perseroan (Persero) PT. Bank Mandiri,Kota Adm. Jakarta Barat,Bachelor,"Teknik Informatika, Statistika, Sistem Informa...",$25,5,5,...,No,No,No,3 to 10,3 to 10,21 to 50,11 - 25%,Yes,No,No
5482,a2418ce4-fd92-4900-bb3d-1bc148b19548,2026-07-16T10:56:52+07:00,Sales Generalis,Perusahaan Perseroan (Persero) Bank Negara Ind...,Kab. Tuban,"Diploma, Bachelor, Profession","Pemasaran Digital, Administrasi Keuangan Dan P...",Mempelajari manajemen pipeline sesuai dengan p...,5,5,...,No,No,No,3 to 10,3 to 10,21 to 50,11 - 25%,Yes,Yes,Yes
20360,a23f4c83-f227-4604-9f29-5613db9b2da3,2026-07-16T12:34:43+07:00,PENGELOLA KEUANGAN DAN ANGGARAN,KANIM KELAS II TPI SIBOLGA,Kota Sibolga,Bachelor,Akuntansi,1. Menyusun rencana kebutuhan anggaran tahunan...,5,1,...,No,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No
27293,a2418853-7b30-4905-8c2b-6c395e36a52d,2026-07-16T12:37:37+07:00,Asisten Pengelola Keuangan,BPS Provinsi Jawa Timur,Kota Surabaya,"Diploma, Bachelor","Manajemen, Ekonomi, Akuntansi",Membantu proses pengelolaan administrasi keuan...,5,2,...,No,No,No,1 to 2,1 to 2,21 to 50,0 - 10%,Yes,Yes,No
24092,a242fa29-19c7-4b62-8f8b-54bdaae85f07,2026-07-16T12:33:05+07:00,PENGELOLA KEUANGAN DAN ANGGARAN,KANIM KELAS I NON TPI KARAWANG,Kab. Karawang,Bachelor,Akuntansi,1. Menyusun rencana kebutuhan anggaran tahunan...,5,1,...,No,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [23]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
2319,a234cc5c-04e7-4e5f-82d4-95c23c6d5c8a,2026-07-16T10:55:27+07:00,L&D Specialist,PT Perkasa Internusa Mandiri,Kota Tangerang,Bachelor,"Seni Rupa, Multimedia, Animasi, Desain Komunik...",Memahami fungsi L&D dan proses pelatihan di pe...,5,1,...,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,No,No,No
4018,a22cf603-afd9-44ce-b90f-d65e25fb9418,2026-07-16T11:20:09+07:00,Teknisi Elektromedik,Yayasan Rumah Sakit Islam Sumatera Barat,Kab. Pasaman Barat,Diploma,Elektromedik,Peserta magang mempelajari struktur organisasi...,6,1,...,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,No,Yes,No,No
1607,a24176a6-91dc-45e3-8e5b-daf78add1b8b,2026-07-16T10:03:52+07:00,Production Engineer Intern,Hon Chuan Indonesia,Kab. Karawang,Bachelor,"Elektro Mekanika, Teknik Mekanika, Elektronika...",Uunderstand manufacturing production processes...,5,8,...,No,No,3 to 10,3 to 10,11 to 20,26 - 50%,Yes,No,No,No
258,a23f3bb0-070e-4ebe-97be-c86d62c08c06,2026-07-16T11:56:48+07:00,Psikiater,LEMBAGA PEMASYARAKATAN PEREMPUAN KELAS III PALU,Kab. Sigi,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,1 to 2,1 to 2,0 to 5,26 - 50%,Yes,No,No,No
5972,a243659e-b9ba-45b5-88bf-dd3e5c09c445,2026-07-16T12:37:02+07:00,Bidang Hubungan Masyarakat (Humas),Kantor Pencarian dan Pertolongan Kelas A Balik...,Kota Balikpapan,Bachelor,"Multimedia, Jurnalistik, Hubungan Masyarakat, ...",Program pemagangan ini dirancang untuk membeka...,5,1,...,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,No,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [130]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage', 'job_category',
       'allows_it_and_computer_majors', 'allows_engineering_majors',
       'allows_business_majors', 'allows_health_majors',
       'allows_science_and_math_majors',
       'allows_agriculture_and_fisheries_majors',
       'allows_arts_and_media_majors', 'allows_social_and_law_majors',
       'allows_education_majors', 'allows_tourism_and_hospitality_majors',
       'requested_quota_category', 'approved_quota_category',
       'applicant_count_category', 'acceptance_percentage_category',
       'allows_bachelor_level', 'allows_diploma_level',
       'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [131]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "job_category",
    "company",
    "regency_city",
    "province",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "allowed_major",
    "allows_it_and_computer_majors",
    "allows_engineering_majors",
    "allows_business_majors",
    "allows_health_majors",
    "allows_science_and_math_majors",
    "allows_agriculture_and_fisheries_majors",
    "allows_arts_and_media_majors",
    "allows_social_and_law_majors",
    "allows_education_majors",
    "allows_tourism_and_hospitality_majors",
    "allows_all_majors",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,job_category,company,regency_city,province,allows_bachelor_level,allows_diploma_level,allows_profession_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
7141,a23529e4-44bc-4e4f-ac21-4e8abc78c000,2026-07-16T10:24:27+07:00,Staff Accounting,Finance & Banking,PT. Wahana Prestasi Logistik,Kota Tangerang Selatan,Banten,Yes,No,No,...,- Membantu proses pencatatan transaksi keuanga...,6,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,5,16.67
23372,a242ffe3-7310-4649-97d5-b8e8730c11d5,2026-07-16T12:33:05+07:00,ASISTEN PENGEMBANGAN WEB,IT & Data,KANIM KELAS I NON TPI PATI,Kab. Pati,Jawa Tengah,Yes,No,No,...,1. Menyusun dan mengelola prosedur kerja serta...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,14,6.67
4774,a23f6282-8f15-456c-86d7-da2b6f32b8ea,2026-07-16T10:19:02+07:00,Intern Maintenance,Engineering & Maintenance,Southeast Asia Pipe Industries,Kab. Lampung Selatan,Lampung,Yes,Yes,No,...,-Membantu tim Maintenance dalam dukungan tekni...,5,1 to 2,1 to 2,6 to 10,11 - 25%,2,2,8,22.22
17174,a2436da2-4f0b-4c2e-8ae1-9d247f44e753,2026-07-16T12:56:11+07:00,PENGELOLA SDM,"HR, Admin & Management",LEMBAGA PEMASYARAKATAN PEREMPUAN KELAS IIA JAK...,Kota Adm. Jakarta Timur,DKI Jakarta,Yes,No,No,...,1. Mengumpulkan data dan informasi yang releva...,5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00
27578,a23f658f-cd26-4c89-becd-14bdc55cb08b,2026-07-16T12:36:05+07:00,Asisten Arsiparis,"Education, Training & Government",BPS Kabupaten Tanah Datar,Kab. Tanah Datar,Sumatera Barat,Yes,No,No,...,"Mendukung penyusunan, pengelolaan, penyimpanan...",5,1 to 2,1 to 2,21 to 50,0 - 10%,1,1,27,3.57
2982,a240ea32-691b-460f-ab46-cd405911bf85,2026-07-16T12:48:47+07:00,Pengelola Kegiatan Kerja,Correctional & Social Services,LEMBAGA PEMASYARAKATAN KELAS III RANGKASBITUNG,Kab. Lebak,Banten,Yes,No,No,...,$28,6,1 to 2,1 to 2,6 to 10,11 - 25%,2,2,7,25.00
11960,a241f2a4-1857-491f-ad2f-3114589b7f05,2026-07-16T20:15:50+07:00,Assistant Producer - IDX,Other,Mnc Televisi Network,Kota Adm. Jakarta Pusat,DKI Jakarta,Yes,No,No,...,$26,5,1 to 2,1 to 2,11 to 20,11 - 25%,2,2,13,14.29
3416,a241459a-f8ea-4374-876f-3b19209feb73,2026-07-16T16:24:19+07:00,Paper Finishing Apprenticeship,"Manufacturing, QA & Production",PT Bukit Muria Jaya,Kab. Karawang,Jawa Barat,Yes,No,No,...,Memahami Proses End-to-End Area Paper Finishin...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,4,20.00
9181,a2419589-ac83-414c-8bcf-e21986f569a2,2026-07-16T10:56:17+07:00,Staff Supply planning,"HR, Admin & Management",Cahaya Abadi Plastik,Kab. Bekasi,Jawa Barat,Yes,Yes,No,...,Merencanakan dan mengendalikan kebutuhan bahan...,5,3 to 10,3 to 10,21 to 50,11 - 25%,4,4,23,16.67
24759,a2415b69-1457-4432-b7b2-c1f5b9cf8741,2026-07-16T12:08:23+07:00,Staf Komunikasi Dan Kesekretariatan,"Media, PR & Creative",BPJS Kesehatan Kantor Cabang Solok,Kota Solok,Sumatera Barat,Yes,Yes,No,...,Membantu melaksanakan kegiatan administratif d...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,16,5.88


In [132]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 